## 1. Imports

In [ ]:
!pip install torch==2.4.1 --index-url https://download.pytorch.org/whl/cu121
!pip install torch_cluster==1.6.3+pt24cu121 --find-links https://data.pyg.org/whl/torch-2.4.0+cu121.html
!pip install torch_geometric==2.5.3
!pip install pyg-lib==0.4.0+pt24cu121 --find-links https://data.pyg.org/whl/torch-2.4.0+cu121.html
!pip install torch_scatter==2.1.2+pt24cu121 --find-links https://data.pyg.org/whl/torch-2.4.0+cu121.html
!pip install torch_sparse==0.6.18+pt24cu121 --find-links https://data.pyg.org/whl/torch-2.4.0+cu121.html
!pip install torch_spline_conv==1.2.2+pt24cu121 --find-links https://data.pyg.org/whl/torch-2.4.0+cu121.html

In [ ]:
!pip install fairchem-core==1.10.0

In [ ]:
!git clone https://github.com/hoang-hoa-tham-hs/fairchem-tio2-s2ef.git
!pip install ase lmdb wandb numba

In [ ]:
%cd /content/fairchem-tio2-s2ef

In [ ]:
import sys
import torch
import os
# sys.path.append(r"D:\Train_thử\fairchem-tio2-s2ef")

import sys
sys.path.append(r"content\fairchem-tio2-s2ef")
from ocpmodels.trainers import ForcesTrainer
from ocpmodels import models
from ocpmodels.common import logger
from ocpmodels.common.utils import setup_logging

import pandas as pd
import numpy as np
from ase.io import read, write
from ase.optimize import LBFGS
from ocpmodels.common.relaxation.ase_utils import OCPCalculator

from ocpmodels.preprocessing import AtomsToGraphs
from ocpmodels.datasets import SinglePointLmdbDataset, TrajectoryLmdbDataset, OC22LmdbDataset, LmdbDataset
import ase.io
from ase.build import bulk
from ase.build import fcc100, add_adsorbate, molecule
from ase.constraints import FixAtoms
from ase.calculators.emt import EMT
from ase.optimize import BFGS
import matplotlib.pyplot as plt
import lmdb
import pickle
from tqdm import tqdm
import torch
import os

from google.colab import drive
drive.mount('/content/drive')

setup_logging()

## 2. Rebuild Slab

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 1: REBUILD SLAB
# ══════════════════════════════════════════════════════════════════
import numpy as np
import os
from ase import Atoms
from ase.io import read, write
from ase.constraints import FixAtoms
from scipy.spatial.distance import cdist

# ── Step 1: Rebuild TiO2 rutile unit cell ─────────────────────────
a, c, u = 4.594, 2.959, 0.305

ti_pos = np.array([[0.0, 0.0, 0.0], [0.5, 0.5, 0.5]])
o_pos  = np.array([[u, u, 0.0], [1-u, 1-u, 0.0],
                   [0.5+u, 0.5-u, 0.5], [0.5-u, 0.5+u, 0.5]])
lattice = np.array([[a, 0, 0], [0, a, 0], [0, 0, c]])
cart    = np.vstack([ti_pos, o_pos]) @ lattice

tio2_rutile = Atoms(numbers=[22,22,8,8,8,8],
                    positions=cart, cell=lattice, pbc=True)

# ── Step 2: Build slab → supercell ────────────────────────────────
from ase.build import surface

slab = surface(tio2_rutile, indices=(1,1,1), layers=4, vacuum=15.0)
slab = slab.repeat([3, 3, 1])

# ── Step 3: Assign tags ───────────────────────────────────────────
positions = slab.get_positions()
numbers   = slab.get_atomic_numbers()
z_coords  = positions[:, 2]
z_rounded = np.round(z_coords, 1)
unique_z  = np.sort(np.unique(z_rounded))[::-1]
top_2_z   = set(unique_z[:2])
tags = np.array([1 if round(z,1) in top_2_z else 0 for z in z_coords])
slab.set_tags(tags)

# ── Freeze bulk atoms ─────────────────────────────────────────────
fixed_indices = [i for i, t in enumerate(tags) if t == 0]
slab.set_constraint(FixAtoms(indices=fixed_indices))
slab.center(vacuum=13.0, axis=2)
slab.set_pbc(True)

# ── Step 4: Find Ti5c sites ───────────────────────────────────────
def find_ti5c(slab, cutoff=3.0):
    pos    = slab.get_positions()
    nums   = slab.get_atomic_numbers()
    tags   = slab.get_tags()
    dists  = cdist(pos, pos)
    o_mask = nums == 8
    ti5c   = []
    for i in np.where((nums == 22) & (tags == 1))[0]:
        n_o = np.sum((dists[i] <= cutoff) & (dists[i] > 0) & o_mask)
        if n_o == 5:
            ti5c.append({"idx": i, "pos": pos[i].copy(), "z": pos[i,2]})
    return ti5c

ti5c_list    = find_ti5c(slab, cutoff=3.0)
best_ti5c    = max(ti5c_list, key=lambda x: x["z"])
all_ti5c_pos = np.array([t["pos"] for t in ti5c_list])

## 3. Load Clean Slab

In [ ]:
import numpy as np
import os
from ase.io import read, write
from scipy.spatial.distance import cdist

# ── Create output directory first ────────────────────────────────
os.makedirs("data", exist_ok=True)

# ── Load your existing clean slab ────────────────────────────────
clean = slab.copy()   # slab is already in memory from Cell 1

# ── Save clean slab for reference ────────────────────────────────
write("data/tio2_clean.traj", clean)
print(f"Clean slab: {len(clean)} atoms")
print(f"Species   : {set(clean.get_chemical_symbols())}")

## 4. Find The Ti5C Locations

In [ ]:
import numpy as np
import os
from ase.io import read, write
from scipy.spatial.distance import cdist

os.makedirs("data", exist_ok=True)

# ── Use slab already in memory ────────────────────────────────────
clean = slab.copy()

pos  = clean.get_positions()
nums = clean.get_atomic_numbers()
syms = np.array(clean.get_chemical_symbols())
tags = clean.get_tags()

# ── Sort all Ti5c sites by z (highest first) ──────────────────────
ti5c_sorted = sorted(ti5c_list, key=lambda x: x["z"], reverse=True)

print(f"Total Ti5c sites found : {len(ti5c_sorted)}")
print(f"\n{'Rank':<5} {'atom_idx':<10} {'x':>7} {'y':>7} {'z':>7}")
print("-" * 40)
for i, site in enumerate(ti5c_sorted):
    print(f"  {i:<4} {site['idx']:<10} "
          f"{site['pos'][0]:>7.3f} "
          f"{site['pos'][1]:>7.3f} "
          f"{site['pos'][2]:>7.3f}")

## 5. Identify The Locations Doping

In [ ]:
from itertools import combinations
import pandas as pd
import numpy as np

# ── All 4 substitution sites ──────────────────────────────────────
sites = [
    {"rank": 0, "ti5c_idx": 19,  "o_idx": 20,  "o_pos": (3.438, 2.542, 20.897)},
    {"rank": 1, "ti5c_idx": 43,  "o_idx": 44,  "o_pos": (5.040, 7.766, 20.897)},
    {"rank": 2, "ti5c_idx": 91,  "o_idx": 92,  "o_pos": (8.902, 2.542, 20.897)},
    {"rank": 3, "ti5c_idx": 115, "o_idx": 116, "o_pos": (10.505,7.766, 20.897)},
]

o_indices    = [s["o_idx"]    for s in sites]
ti5c_indices = [s["ti5c_idx"] for s in sites]

# ── Generate ALL combinations including single ────────────────────
all_combos = {}

for r in [1, 2, 3, 4]:                                    # ← added 1
    label = {1: "single", 2: "pair", 3: "triple", 4: "quad"}[r]
    for combo in combinations(range(len(sites)), r):
        o_combo  = tuple(sites[i]["o_idx"]    for i in combo)
        ti_combo = tuple(sites[i]["ti5c_idx"] for i in combo)
        key      = "N" + "_".join(str(o) for o in o_combo)
        all_combos[key] = {
            "label":        label,
            "site_ranks":   combo,
            "o_indices":    o_combo,
            "ti5c_indices": ti_combo,
            "n_sites":      r,
        }

# ── Print summary table ───────────────────────────────────────────
print(f"{'Key':<25} {'Type':<8} {'O indices':<25} {'Ti5c indices'}")
print("-" * 80)
for key, info in all_combos.items():
    print(f"  {key:<23} {info['label']:<8} "
          f"{str(info['o_indices']):<25} "
          f"{str(info['ti5c_indices'])}")

print(f"\nTotal combinations : {len(all_combos)}")
print(f"  Singles : {sum(1 for v in all_combos.values() if v['n_sites']==1)}")
print(f"  Pairs   : {sum(1 for v in all_combos.values() if v['n_sites']==2)}")
print(f"  Triples : {sum(1 for v in all_combos.values() if v['n_sites']==3)}")
print(f"  Quads   : {sum(1 for v in all_combos.values() if v['n_sites']==4)}")

## 6. Dope N

In [ ]:
from itertools import combinations
import pandas as pd
import numpy as np

# ── All 4 substitution sites ──────────────────────────────────────
sites = [
    {"rank": 0, "ti5c_idx": 19,  "o_idx": 20,  "o_pos": (3.438, 2.542, 20.897)},
    {"rank": 1, "ti5c_idx": 43,  "o_idx": 44,  "o_pos": (5.040, 7.766, 20.897)},
    {"rank": 2, "ti5c_idx": 91,  "o_idx": 92,  "o_pos": (8.902, 2.542, 20.897)},
    {"rank": 3, "ti5c_idx": 115, "o_idx": 116, "o_pos": (10.505,7.766, 20.897)},
]

o_indices    = [s["o_idx"]    for s in sites]
ti5c_indices = [s["ti5c_idx"] for s in sites]

# ── Generate ALL combinations including single ────────────────────
all_combos = {}

for r in [1, 2, 3, 4]:                                    # ← added 1
    label = {1: "single", 2: "pair", 3: "triple", 4: "quad"}[r]
    for combo in combinations(range(len(sites)), r):
        o_combo  = tuple(sites[i]["o_idx"]    for i in combo)
        ti_combo = tuple(sites[i]["ti5c_idx"] for i in combo)
        key      = "N" + "_".join(str(o) for o in o_combo)
        all_combos[key] = {
            "label":        label,
            "site_ranks":   combo,
            "o_indices":    o_combo,
            "ti5c_indices": ti_combo,
            "n_sites":      r,
        }

# ── Print summary table ───────────────────────────────────────────
print(f"{'Key':<25} {'Type':<8} {'O indices':<25} {'Ti5c indices'}")
print("-" * 80)
for key, info in all_combos.items():
    print(f"  {key:<23} {info['label']:<8} "
          f"{str(info['o_indices']):<25} "
          f"{str(info['ti5c_indices'])}")

print(f"\nTotal combinations : {len(all_combos)}")
print(f"  Singles : {sum(1 for v in all_combos.values() if v['n_sites']==1)}")
print(f"  Pairs   : {sum(1 for v in all_combos.values() if v['n_sites']==2)}")
print(f"  Triples : {sum(1 for v in all_combos.values() if v['n_sites']==3)}")
print(f"  Quads   : {sum(1 for v in all_combos.values() if v['n_sites']==4)}")

## 7. Run Relaxation To Calculate E_slab For New N-doped Slabs

In [ ]:
from ase.optimize import LBFGS
from ase.io import read, write
from ocpmodels.common.relaxation.ase_utils import OCPCalculator
import numpy as np
import pandas as pd

calc = OCPCalculator(
    config_yml = r"/content/fairchem-tio2-s2ef/configs/s2ef/all/gemnet/gemnet-dT.yml",
    checkpoint = r"/content/drive/MyDrive/abc/checkpoint.pt",
)

relax_results = []

for key, info in all_combos.items():
    doped     = combo_slabs[key]
    traj_path = f"data/tio2_{key}_relax.traj"

    print(f"\n{'='*55}")
    print(f"  {info['label'].upper()} : {key}")
    print(f"  O→N : {info['o_indices']}")
    print(f"{'='*55}")

    doped.calc = calc

    dyn = LBFGS(doped, trajectory=traj_path, maxstep=0.05)
    dyn.run(fmax=0.05, steps=100)

    traj       = read(traj_path, ":")
    e_init     = traj[0].get_potential_energy()
    e_final    = traj[-1].get_potential_energy()
    fmax_init  = np.linalg.norm(traj[0].get_forces(),  axis=1).max()
    fmax_final = np.linalg.norm(traj[-1].get_forces(), axis=1).max()
    converged  = fmax_final <= 0.05

    print(f"  Frames     : {len(traj)}")
    print(f"  E_initial  : {e_init:.4f} eV")
    print(f"  E_final    : {e_final:.4f} eV")
    print(f"  ΔE         : {e_final - e_init:.4f} eV")
    print(f"  fmax_final : {fmax_final:.4f} eV/Å")
    print(f"  Converged  : {'YES ✓' if converged else 'NO ✗'}")

    write(f"data/tio2_{key}_relaxed.xsf", traj[-1])

    relax_results.append({
        "key":            key,
        "type":           info["label"],
        "n_sites":        info["n_sites"],
        "o_indices":      str(info["o_indices"]),
        "ti5c_indices":   str(info["ti5c_indices"]),
        "n_atoms":        len(doped),
        "n_frames":       len(traj),
        "E_initial (eV)": round(e_init,           4),
        "E_final (eV)":   round(e_final,           4),
        "delta_E (eV)":   round(e_final - e_init,  4),
        "fmax_initial":   round(fmax_init,          4),
        "fmax_final":     round(fmax_final,         4),
        "converged":      converged,
        "traj_file":      traj_path,
    })

df_relax = pd.DataFrame(relax_results)

# ── Save ──────────────────────────────────────────────────────────
df_relax.to_csv("data/all_relaxation_summary.csv", index=False)   # ← renamed

# ── Print grouped by type ─────────────────────────────────────────
print(f"\n{'='*55}")
print(f"ALL COMBINATIONS COMPLETE")
print(f"{'='*55}")

for n, label in [(1,"SINGLE"), (2,"PAIRS"), (3,"TRIPLES"), (4,"QUAD")]:
    subset = df_relax[df_relax["n_sites"] == n]
    if subset.empty:
        continue
    print(f"\n  ── {label} ──")
    print(subset[["key","type","E_final (eV)",
                  "fmax_final","converged"]].to_string(index=False))

print(f"\nSaved → data/all_relaxation_summary.csv")

## 8. Check Displacement Of Ti5c Locations After Doping

In [ ]:
import numpy as np
import pandas as pd
from ase.io import read

clean_pos = clean.get_positions()
disp_rows = []

print(f"\n{'='*90}")
print(f"  Ti₅c Displacement — All Combinations (Single / Pair / Triple / Quad)")
print(f"{'='*90}")

for key, info in all_combos.items():
    traj_path = f"data/tio2_{key}_relax.traj"

    try:
        relaxed           = read(traj_path, "-1")
        shift             = clean.get_center_of_mass() - relaxed.get_center_of_mass()
        doped_pos_aligned = relaxed.get_positions() + shift

        print(f"\n  [{info['label'].upper()}] {key}")
        print(f"  {'Ti5c_idx':<10} {'N_idx':<8} {'Δx':>8} {'Δy':>8} {'Δz':>8} {'|Δr|':>8}  Status")
        print(f"  {'-'*65}")

        for rank in info["site_ranks"]:
            ti5c_idx = sites[rank]["ti5c_idx"]
            n_idx    = sites[rank]["o_idx"]

            disp_vec = doped_pos_aligned[ti5c_idx] - clean_pos[ti5c_idx]
            disp_mag = np.linalg.norm(disp_vec)
            disp_z   = disp_vec[2]
            disp_xy  = np.linalg.norm(disp_vec[:2])
            status   = "✓" if abs(disp_z - 0.4) < 0.2 else "⚠"

            print(f"  {ti5c_idx:<10} {n_idx:<8} "
                  f"{disp_vec[0]:>8.4f} "
                  f"{disp_vec[1]:>8.4f} "
                  f"{disp_z:>8.4f} "
                  f"{disp_mag:>8.4f}  {status}")

            disp_rows.append({
                "combo_key":  key,
                "type":       info["label"],
                "n_sites":    info["n_sites"],
                "o_indices":  str(info["o_indices"]),
                "ti5c_idx":   ti5c_idx,
                "n_idx":      n_idx,
                "delta_x":    round(disp_vec[0], 4),
                "delta_y":    round(disp_vec[1], 4),
                "delta_z":    round(disp_z,      4),
                "delta_xy":   round(disp_xy,     4),
                "delta_mag":  round(disp_mag,    4),
                "status":     status,
            })

    except FileNotFoundError:
        print(f"  ⚠ {key} — traj not found, skipped")

df_disp = pd.DataFrame(disp_rows)

# ── Save ──────────────────────────────────────────────────────────
df_disp.to_csv("data/all_displacement_summary.csv", index=False)  # ← renamed

# ── Print grouped summary ─────────────────────────────────────────
print(f"\n{'='*90}")
print(f"  Displacement Summary by Type")
print(f"{'='*90}")

for n, label in [(1,"SINGLE"), (2,"PAIRS"), (3,"TRIPLES"), (4,"QUAD")]:
    subset = df_disp[df_disp["n_sites"] == n]
    if subset.empty:
        continue
    print(f"\n  ── {label} ──")
    print(f"  {'combo_key':<25} {'ti5c_idx':<10} {'n_idx':<8} "
          f"{'Δz':>8} {'|Δr|':>8}  Status")
    print(f"  {'-'*70}")
    for _, row in subset.iterrows():
        print(f"  {row['combo_key']:<25} {row['ti5c_idx']:<10} {row['n_idx']:<8} "
              f"{row['delta_z']:>8.4f} {row['delta_mag']:>8.4f}  {row['status']}")

print(f"\nTotal rows saved : {len(df_disp)}")
print(f"Saved → data/all_displacement_summary.csv")

## 9. Summarize The Results

In [ ]:
import pandas as pd
import numpy as np

# ── Load from renamed CSVs (now include singles) ──────────────────
df_relax = pd.read_csv("data/all_relaxation_summary.csv")    # ← renamed
df_disp  = pd.read_csv("data/all_displacement_summary.csv")  # ← renamed

# ── Per-combo displacement stats ──────────────────────────────────
disp_stats = df_disp.groupby("combo_key").agg(
    mean_dz = ("delta_z",   "mean"),
    max_dz  = ("delta_z",   "max"),
    min_dz  = ("delta_z",   "min"),
    mean_dr = ("delta_mag", "mean"),
    max_dr  = ("delta_mag", "max"),
).reset_index().rename(columns={"combo_key": "key"})

# ── Merge relaxation + displacement stats ─────────────────────────
master = df_relax.merge(disp_stats, on="key", how="left")

# Sort by n_sites then E_final
master = master.sort_values(["n_sites", "E_final (eV)"]).reset_index(drop=True)

# ── Save master table ─────────────────────────────────────────────
master.to_csv("data/master_comparison_table.csv", index=False)

# ── Display grouped by n_sites ────────────────────────────────────
display_cols = [
    "type", "key", "n_sites",
    "E_final (eV)", "delta_E (eV)",
    "fmax_final", "converged",
    "mean_dz", "max_dz", "mean_dr",
]

print(f"\n{'='*120}")
print(f"  MASTER COMPARISON TABLE — All N-doping Combinations")
print(f"{'='*120}")

for n, label in [(1,"SINGLE"), (2,"PAIRS"), (3,"TRIPLES"), (4,"QUAD")]:
    subset = master[master["n_sites"] == n]
    if subset.empty:
        continue
    print(f"\n  ── {label} ({len(subset)} combinations) ──")
    print(subset[display_cols].to_string(index=False))

print(f"\nSaved → data/master_comparison_table.csv")

# ── Summary by type ───────────────────────────────────────────────
type_order   = {"single": 1, "pair": 2, "triple": 3, "quad": 4}
summary = master.groupby("type").agg(
    count        = ("key",          "count"),
    best_E       = ("E_final (eV)", "min"),
    worst_E      = ("E_final (eV)", "max"),
    mean_delta_E = ("delta_E (eV)", "mean"),
    mean_dz      = ("mean_dz",      "mean"),
    n_converged  = ("converged",    "sum"),
).reset_index()

summary["conv_rate (%)"] = (
    summary["n_converged"] / summary["count"] * 100
).round(1)
summary["_order"] = summary["type"].map(type_order)
summary = summary.sort_values("_order").drop(columns="_order").reset_index(drop=True)

summary.to_csv("data/summary_by_type.csv", index=False)

print(f"\n{'='*80}")
print(f"  Summary by combination type")
print(f"{'='*80}")
print(summary.to_string(index=False))
print(f"\nSaved → data/summary_by_type.csv")

## 10. Calculation E_adsorbate Of 75 N-doped Slabs + Fragments

## 10.1 Load The Data

In [ ]:
import numpy as np
import pandas as pd
import os
from ase.io import read, write
from ase.optimize import LBFGS
from ase.constraints import FixAtoms
from ocpmodels.common.relaxation.ase_utils import OCPCalculator

os.makedirs("data/adsorption", exist_ok=True)

calc = OCPCalculator(
    config_yml = r"/content/fairchem-tio2-s2ef/configs/s2ef/all/gemnet/gemnet-dT.yml",
    checkpoint = r"/content/drive/MyDrive/abc/checkpoint.pt",
)

ALL_PROXIES = {
    "OH":  {"label": "Hydroxyl –OH",   "represents": "H2O2",   "group": "h2o2"},
    "H2O": {"label": "H₂O product",    "represents": "H2O2",   "group": "h2o2"},
    "CHO": {"label": "Aldehyde –CHO",  "represents": "Glucose", "group": "glucose"},
    "CO2": {"label": "CO₂ product",    "represents": "Glucose", "group": "glucose"},
    "O":   {"label": "Atomic O",       "represents": "H2O2",   "group": "h2o2"},
}

FRAG_TRAJS = {
    "OH":  "/content/tio2_OH_relax.traj",
    "H2O": "/content/tio2_H2O_relax.traj",
    "CHO": "/content/tio2_CHO_relax.traj",
    "CO2": "/content/tio2_CO2_relax.traj",
    "O":   "/content/tio2_O_relax.traj",
}

# ── Helper: read best valid frame ─────────────────────────────────
def read_best_valid_frame(traj_path):
    """
    Read all frames, return the one with lowest fmax
    that has finite energy and forces.
    """
    try:
        frames = read(traj_path, ":")
    except FileNotFoundError:
        return None, None, None

    valid = []
    for f in frames:
        try:
            e      = f.get_potential_energy()
            forces = f.get_forces()
            if np.isfinite(e) and np.all(np.isfinite(forces)):
                valid.append(f)
        except Exception:
            pass

    if not valid:
        return None, None, None

    fmaxes = [np.linalg.norm(f.get_forces(), axis=1).max() for f in valid]
    best   = valid[int(np.argmin(fmaxes))]
    return best, best.get_potential_energy(), min(fmaxes)

# ── Clean slab energy ─────────────────────────────────────────────
print("Loading slab energies...")
clean_slab = read("/content/tio2_clean.traj", "-1")
clean_slab.calc = calc
E_slab_clean = clean_slab.get_potential_energy()
print(f"E_slab_clean : {E_slab_clean:.4f} eV")

# ── All N-doped slab energies ─────────────────────────────────────
master   = pd.read_csv("/content/master_comparison_table.csv")
E_slab   = {"pristine": E_slab_clean}
slab_obj = {"pristine": clean_slab}

print(f"\nLoading {len(master)} N-doped slabs...")
for _, row in master.iterrows():
    key       = row["key"]
    traj_path = f"/content/tio2_{key}_relax.traj"
    try:
        slab = read(traj_path, "-1")
        slab.calc = calc
        e    = slab.get_potential_energy()
        E_slab[key]   = e
        slab_obj[key] = slab
        print(f"  ✓ {key:<25} E={e:.4f} eV  [{row['type']}]")
    except FileNotFoundError:
        print(f"  ⚠ {key} — not found, skipped")

print(f"\nTotal slabs loaded : {len(E_slab)}")
print(f"  Clean   : 1")
print(f"  N-doped : {len(E_slab)-1}")

## 10.2 Load Fragments And E_Slabs Before

In [ ]:
import numpy as np
import pandas as pd
import os
import math
from ase import Atoms
from ase.io import read, write
from ase.optimize import LBFGS
from ase.constraints import FixAtoms
from ocpmodels.common.relaxation.ase_utils import OCPCalculator

os.makedirs("data/adsorption", exist_ok=True)

calc = OCPCalculator(
    config_yml = r"/content/fairchem-tio2-s2ef/configs/s2ef/all/gemnet/gemnet-dT.yml",
    checkpoint = r"/content/drive/MyDrive/abc/checkpoint.pt",
)

ALL_PROXIES = {
    "OH":  {"label": "Hydroxyl –OH",   "represents": "H2O2",   "group": "h2o2"},
    "H2O": {"label": "H₂O product",    "represents": "H2O2",   "group": "h2o2"},
    "CHO": {"label": "Aldehyde –CHO",  "represents": "Glucose", "group": "glucose"},
    "CO2": {"label": "CO₂ product",    "represents": "Glucose", "group": "glucose"},
    "O":   {"label": "Atomic O",       "represents": "H2O2",   "group": "h2o2"},
}

FRAG_GEOMETRIES = {
    "CHO": Atoms("CHO", positions=[
        [0.000,  0.000, 0.000],
        [1.100,  0.000, 0.000],
        [-0.550, 0.950, 0.000],
    ]),
    "OH": Atoms("OH", positions=[
        [0.000, 0.000, 0.000],
        [0.000, 0.000, 0.970],
    ]),
    "CO2": Atoms("OCO", positions=[
        [0.000, 0.000, -1.160],
        [0.000, 0.000,  0.000],
        [0.000, 0.000,  1.160],
    ]),
    "H2O": Atoms("HOH", positions=[
        [-0.960*math.sin(math.radians(52.25)), 0,
          0.960*math.cos(math.radians(52.25))],
        [0.000, 0.000, 0.000],
        [ 0.960*math.sin(math.radians(52.25)), 0,
          0.960*math.cos(math.radians(52.25))],
    ]),
    "O": Atoms("O", positions=[[0.000, 0.000, 0.000]]),
}

master   = pd.read_csv("/content/master_comparison_table.csv")
E_slab   = {}
slab_obj = {}

# ── Load clean slab ───────────────────────────────────────────────
clean_slab = read("/content/tio2_clean.traj", "-1")
clean_slab.calc = calc
E_slab["pristine"]   = clean_slab.get_potential_energy()
slab_obj["pristine"] = clean_slab
print(f"E_slab_clean : {E_slab['pristine']:.4f} eV")

# ── Load all N-doped slabs ────────────────────────────────────────
for _, row in master.iterrows():
    key       = row["key"]
    traj_path = f"/content/tio2_{key}_relax.traj"
    try:
        slab = read(traj_path, "-1")
        slab.calc = calc
        E_slab[key]   = slab.get_potential_energy()
        slab_obj[key] = slab
        print(f"  ✓ {key:<25} E={E_slab[key]:.4f} eV")
    except FileNotFoundError:
        print(f"  ⚠ {key} not found")

# ── Build ordered slab list (exclude pristine) ───────────────────
ndoped_keys = master.sort_values(
    ["n_sites","key"])["key"].tolist()
ndoped_keys = [k for k in ndoped_keys if k in E_slab]

print(f"\nTotal N-doped slabs : {len(ndoped_keys)}")
for i, k in enumerate(ndoped_keys):
    print(f"  [{i:>2}] {k}")

# ── Helpers ───────────────────────────────────────────────────────
def read_best_valid_frame(traj_path):
    try:
        frames = read(traj_path, ":")
    except FileNotFoundError:
        return None, None, None
    valid = []
    for f in frames:
        try:
            e = f.get_potential_energy()
            forces = f.get_forces()
            if np.isfinite(e) and np.all(np.isfinite(forces)):
                valid.append(f)
        except Exception:
            pass
    if not valid:
        return None, None, None
    fmaxes = [np.linalg.norm(f.get_forces(), axis=1).max()
               for f in valid]
    best = valid[int(np.argmin(fmaxes))]
    return best, best.get_potential_energy(), min(fmaxes)

def place_fragment_above_ti5c(slab, frag_geom, height=2.5):
    slab_copy = slab.copy()
    slab_pos  = slab_copy.get_positions()
    slab_nums = slab_copy.get_atomic_numbers()
    ti_mask   = slab_nums == 22
    ti5c_idx  = np.where(ti_mask)[0][
                    np.argmax(slab_pos[ti_mask, 2])]
    ti5c_pos  = slab_pos[ti5c_idx]
    frag      = frag_geom.copy()
    frag_pos  = frag.get_positions()
    lowest_i  = np.argmin(frag_pos[:, 2])
    frag.translate([0, 0, -frag_pos[lowest_i, 2]])
    com = frag.get_center_of_mass()
    frag.translate([
        ti5c_pos[0] - com[0],
        ti5c_pos[1] - com[1],
        ti5c_pos[2] + height - com[2],
    ])
    frag.set_tags([2] * len(frag))
    system = slab_copy + frag
    system.set_cell(slab_copy.get_cell())
    system.set_pbc(True)
    fix_idx = [i for i, t in enumerate(system.get_tags()) if t == 0]
    system.set_constraint(FixAtoms(indices=fix_idx))
    return system

def relax_system(system, traj_path, calc, maxstep=0.04, steps=100):
    system.calc = calc
    try:
        dyn = LBFGS(system, trajectory=traj_path, maxstep=maxstep)
        dyn.run(fmax=0.05, steps=steps)
    except Exception as ex:
        print(f"    ⚠ LBFGS warning: {ex}")
    return read_best_valid_frame(traj_path)

def run_batch(slab_keys_batch, label="batch"):
    """
    Run relaxation for a list of slab keys × all fragments.
    Saves results to data/adsorption/ and returns list of dicts.
    """
    batch_results = []
    total = len(slab_keys_batch) * len(ALL_PROXIES)
    done  = 0

    print(f"\n{'='*60}")
    print(f"  {label}  —  {len(slab_keys_batch)} slabs × "
          f"{len(ALL_PROXIES)} fragments = {total} runs")
    print(f"{'='*60}")

    for slab_key in slab_keys_batch:
        if slab_key not in slab_obj:
            print(f"  ⚠ {slab_key} not in slab_obj, skipped")
            continue

        row       = master[master["key"] == slab_key]
        slab_type = row["type"].values[0]    if len(row) > 0 else "unknown"
        n_sites   = int(row["n_sites"].values[0]) if len(row) > 0 else 0
        slab      = slab_obj[slab_key]

        print(f"\n  [{slab_type.upper()}] {slab_key}")

        for frag_name in ALL_PROXIES:
            done += 1
            traj_path = f"data/adsorption/{slab_key}_{frag_name}_relax.traj"

            # Skip if cached
            if os.path.exists(traj_path):
                frame, e_complex, fmax_v = read_best_valid_frame(traj_path)
                if frame is not None:
                    conv = "✓" if fmax_v <= 0.05 else f"✗({fmax_v:.3f})"
                    print(f"    [{done:>3}/{total}] {frag_name:<5} "
                          f"E={e_complex:.4f}  fmax={fmax_v:.4f}  "
                          f"{conv}  [cached]")
                    batch_results.append({
                        "slab_key":  slab_key,
                        "slab_type": slab_type,
                        "n_sites":   n_sites,
                        "fragment":  frag_name,
                        "e_complex": e_complex,
                        "fmax":      fmax_v,
                    })
                    continue

            # Place and relax
            system = place_fragment_above_ti5c(
                slab, FRAG_GEOMETRIES[frag_name]
            )
            frame, e_complex, fmax_v = relax_system(
                system, traj_path, calc
            )

            if frame is None:
                print(f"    [{done:>3}/{total}] {frag_name:<5} "
                      f"✗ no valid frames")
                continue

            conv = "✓" if fmax_v <= 0.05 else f"✗({fmax_v:.3f})"
            print(f"    [{done:>3}/{total}] {frag_name:<5} "
                  f"E={e_complex:.4f}  fmax={fmax_v:.4f}  {conv}")

            batch_results.append({
                "slab_key":  slab_key,
                "slab_type": slab_type,
                "n_sites":   n_sites,
                "fragment":  frag_name,
                "e_complex": e_complex,
                "fmax":      fmax_v,
            })

    print(f"\n  Batch complete : {len(batch_results)}/{total} successful")
    return batch_results

# ── Initialize collector ──────────────────────────────────────────
ndoped_complex_results = []
print("\nSetup complete — ready to run batches")
print(f"Slabs to process: {len(ndoped_keys)}")
print(f"Batches needed  : {-(-len(ndoped_keys)//5)} "
      f"(5 slabs per batch)")

## 10.3 Run E_complex Of 75 Structures Including 3 Batch

In [ ]:
batch_1 = run_batch(ndoped_keys[0:5], label="Batch 1 / slabs 0-4")
ndoped_complex_results.extend(batch_1)
print(f"\nTotal so far: {len(ndoped_complex_results)} results")
df_batch1 = pd.DataFrame(ndoped_complex_results)
df_batch1.to_csv("data/adsorption/ndoped_complex_checkpoint_1.csv", index=False)
print(f"Saved → data/adsorption/ndoped_complex_checkpoint_1.csv  ({len(df_batch1)} rows)")

In [ ]:
batch_2 = run_batch(ndoped_keys[5:10], label="Batch 2 / slabs 5-9")
ndoped_complex_results.extend(batch_2)
print(f"\nTotal so far: {len(ndoped_complex_results)} results")
df_batch2 = pd.DataFrame(ndoped_complex_results)
df_batch2.to_csv("data/adsorption/ndoped_complex_checkpoint_2.csv", index=False)
print(f"Saved → data/adsorption/ndoped_complex_checkpoint_2.csv  ({len(df_batch1)} rows)")

In [ ]:
batch_3 = run_batch(ndoped_keys[10:15], label="Batch 3 / slabs 10-14")
ndoped_complex_results.extend(batch_3)
print(f"\nTotal so far: {len(ndoped_complex_results)} results")

## 10.4 E_gas Based On Atomic Reference Energies From OC22 



- O = -7.459 eV
- H = -3.386 eV
- C = -7.332 eV
- Using Formula Σᵢ nᵢ · Eᵢʳᵉᶠ
- O = -7.459 eV
- OH = -10.845 eV
- H2O = -14.231 eV
- CO2 = -22.250 eV
- CHO =  -18.177 eV

## 10.5 Calculate E_adsorbate

In [ ]:
# ══════════════════════════════════════════════════════════════════
# E_ad = E_complex − E_slab − E_gas
# ΔE_ad = E_ad(N-doped) − E_ad(pristine)
# E_gas from OC22 atomic reference energies:
#   E_H = −3.386 eV | E_O = −7.459 eV
#   E_C = −7.332 eV | E_N = −8.309 eV
# ══════════════════════════════════════════════════════════════════

# ── OC22 atomic reference energies ───────────────────────────────
E_REF = {
    "H": -3.386,
    "O": -7.459,
    "C": -7.332,
    "N": -8.309,
}

# ── E_gas per fragment: E_gas = Σ nᵢ · Eᵢʳᵉᶠ ────────────────────
E_GAS = {
    "O":   E_REF["O"],                                  # -7.459
    "OH":  E_REF["O"] + E_REF["H"],                     # -10.845
    "H2O": E_REF["O"] + 2 * E_REF["H"],                 # -14.231
    "CO2": E_REF["C"] + 2 * E_REF["O"],                 # -22.250
    "CHO": E_REF["C"] + E_REF["H"] + E_REF["O"],        # -18.177
}

all_results = []

# ── Pristine baseline ─────────────────────────────────────────────
for frag in E_ad_pristine:
    if frag not in E_GAS:
        print(f"  [WARN] No E_gas defined for fragment '{frag}' — skipped")
        continue

    e_gas    = E_GAS[frag]
    e_complex = E_complex_pristine[frag]
    e_ad     = e_complex - E_slab_clean - e_gas

    all_results.append({
        "slab_key":       "pristine",
        "slab_type":      "pristine",
        "n_sites":        0,
        "fragment":       frag,
        "group":          ALL_PROXIES[frag]["group"],
        "represents":     ALL_PROXIES[frag]["represents"],
        "label":          ALL_PROXIES[frag]["label"],
        "E_slab (eV)":    round(E_slab_clean,  4),
        "E_gas (eV)":     round(e_gas,          4),
        "E_complex (eV)": round(e_complex,      4),
        "E_ad (eV)":      round(e_ad,           4),
        "fmax":           round(fmax_pristine[frag], 4),
        "converged":      fmax_pristine[frag] <= 0.05,
        "delta_E_ad":     0.0,
        "enhanced":       False,
    })

# Store pristine E_ad with E_gas for delta reference
E_ad_pristine_with_egas = {
    r["fragment"]: r["E_ad (eV)"]
    for r in all_results
    if r["slab_key"] == "pristine"
}

# ── N-doped results ───────────────────────────────────────────────
for entry in ndoped_complex_results:
    slab_key  = entry["slab_key"]
    frag      = entry["fragment"]
    e_complex = entry["e_complex"]
    fmax_v    = entry["fmax"]

    if slab_key not in E_slab:
        continue
    if frag not in E_ad_pristine_with_egas:
        continue
    if frag not in E_GAS:
        print(f"  [WARN] No E_gas defined for fragment '{frag}' — skipped")
        continue

    e_gas  = E_GAS[frag]
    e_ad   = e_complex - E_slab[slab_key] - e_gas
    delta  = e_ad - E_ad_pristine_with_egas[frag]

    all_results.append({
        "slab_key":       slab_key,
        "slab_type":      entry["slab_type"],
        "n_sites":        entry["n_sites"],
        "fragment":       frag,
        "group":          ALL_PROXIES[frag]["group"],
        "represents":     ALL_PROXIES[frag]["represents"],
        "label":          ALL_PROXIES[frag]["label"],
        "E_slab (eV)":    round(E_slab[slab_key], 4),
        "E_gas (eV)":     round(e_gas,             4),
        "E_complex (eV)": round(e_complex,          4),
        "E_ad (eV)":      round(e_ad,               4),
        "fmax":           round(fmax_v,             4),
        "converged":      fmax_v <= 0.05,
        "delta_E_ad":     round(delta,              4),
        "enhanced":       delta < 0,
    })

df_all = pd.DataFrame(all_results)
df_all.to_csv("data/adsorption/all_adsorption_results.csv", index=False)

# ── Print grouped table ───────────────────────────────────────────
print(f"\n{'='*80}")
print(f"  ADSORPTION RESULTS  —  E_ad = E_complex − E_slab − E_gas")
print(f"  E_gas from OC22 atomic reference energies (Σ nᵢ·Eᵢʳᵉᶠ)")
print(f"  ΔE_ad = E_ad(N-doped) − E_ad(pristine)")
print(f"  Negative ΔE_ad = stronger binding on N-doped surface")
print(f"{'='*80}")

print(f"\n  E_gas reference values used:")
for frag, val in E_GAS.items():
    print(f"    {frag:<6}: {val:+.3f} eV")

for frag in ALL_PROXIES:
    subset = df_all[df_all["fragment"] == frag].copy()
    if subset.empty:
        continue
    print(f"\n  [{frag}] {ALL_PROXIES[frag]['label']}")
    print(f"  E_gas = {E_GAS.get(frag, 'N/A')} eV")
    print(f"  {'Slab':<25} {'Type':<8} {'E_ad':>10} {'ΔE_ad':>9}  Enhanced  Conv")
    print(f"  {'-'*70}")
    for _, r in subset.sort_values("n_sites").iterrows():
        enh  = "✓" if r["enhanced"]  else "—"
        conv = "✓" if r["converged"] else "✗"
        print(f"  {r['slab_key']:<25} {r['slab_type']:<8} "
              f"{r['E_ad (eV)']:>+10.4f} {r['delta_E_ad']:>+9.4f}  "
              f"{enh:<9} {conv}")

print(f"\nSaved → data/adsorption/all_adsorption_results.csv")

## 10.6 Summarize The results

In [ ]:
# ── Group by slab_type + fragment ────────────────────────────────
df_all = pd.read_csv("data/adsorption/all_adsorption_results.csv")

# Verify E_gas column exists (added in updated formula)
if "E_gas (eV)" not in df_all.columns:
    raise ValueError(
        "Column 'E_gas (eV)' not found. "
        "Re-run the cell above with the updated E_ad = E_complex − E_slab − E_gas formula first."
    )

summary = df_all.groupby(["slab_type", "fragment"]).agg(
    count        = ("E_ad (eV)",   "count"),
    E_gas        = ("E_gas (eV)",  "first"),   # constant per fragment
    mean_E_ad    = ("E_ad (eV)",   "mean"),
    best_E_ad    = ("E_ad (eV)",   "min"),
    mean_delta   = ("delta_E_ad",  "mean"),
    best_delta   = ("delta_E_ad",  "min"),
    n_enhanced   = ("enhanced",    "sum"),
    n_converged  = ("converged",   "sum"),
).reset_index()

summary["enhance_rate (%)"] = (
    summary["n_enhanced"] / summary["count"] * 100
).round(1)

summary["converge_rate (%)"] = (
    summary["n_converged"] / summary["count"] * 100
).round(1)

# Sort: pristine first, then single, pair, triple, quad
type_order = {"pristine": 0, "single": 1, "pair": 2, "triple": 3, "quad": 4}
summary["_ord"] = summary["slab_type"].map(type_order).fillna(99)
summary = (
    summary
    .sort_values(["_ord", "fragment"])
    .drop(columns="_ord")
    .reset_index(drop=True)
)

summary.to_csv("data/adsorption/adsorption_summary_by_type.csv", index=False)

# ── Display ───────────────────────────────────────────────────────
pd.set_option("display.float_format", "{:+.4f}".format)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

print(f"\n{'='*110}")
print(f"  SUMMARY TABLE — E_ad = E_complex − E_slab − E_gas (OC22 atomic reference)")
print(f"  ΔE_ad = E_ad(N-doped) − E_ad(pristine) | Negative = stronger binding")
print(f"{'='*110}")

# Print E_gas reference reminder
print(f"\n  E_gas used per fragment:")
egas_ref = (
    df_all[["fragment", "E_gas (eV)"]]
    .drop_duplicates()
    .sort_values("fragment")
)
for _, row in egas_ref.iterrows():
    print(f"    {row['fragment']:<6}: {row['E_gas (eV)']:+.3f} eV")

print()
print(summary.to_string(index=False))

# ── Highlight best configuration per fragment ─────────────────────
print(f"\n{'='*110}")
print(f"  BEST N-DOPED CONFIG per fragment (lowest ΔE_ad, excluding pristine)")
print(f"{'='*110}")

ndoped_only = df_all[df_all["slab_type"] != "pristine"].copy()

if not ndoped_only.empty:
    best_per_frag = (
        ndoped_only
        .sort_values("delta_E_ad")
        .groupby("fragment")
        .first()
        .reset_index()
    )[["fragment", "slab_key", "slab_type", "n_sites",
       "E_gas (eV)", "E_ad (eV)", "delta_E_ad", "fmax", "converged"]]

    print(best_per_frag.to_string(index=False))

print(f"\nSaved → data/adsorption/adsorption_summary_by_type.csv")

## 10.7 Visualize By Chart

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
import numpy as np
from matplotlib.gridspec import GridSpec

df_all = pd.read_csv("data/adsorption/all_adsorption_results.csv")

# ── Style ─────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "font.size":         9,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.linewidth":    0.8,
    "xtick.major.size":  3,
    "ytick.major.size":  3,
    "xtick.major.width": 0.7,
    "ytick.major.width": 0.7,
})

TYPE_CFG = {
    "pristine": {"color": "#6B7280", "label": "Pristine",      "marker": "D"},
    "single":   {"color": "#2563EB", "label": "Single-doped",  "marker": "o"},
    "pair":     {"color": "#16A34A", "label": "Pair-doped",    "marker": "s"},
    "triple":   {"color": "#D97706", "label": "Triple-doped",  "marker": "^"},
    "quad":     {"color": "#DC2626", "label": "Quad-doped",    "marker": "P"},
}

fragments  = list(ALL_PROXIES.keys())
slab_order = ["pristine"] + [k for k in E_slab if k != "pristine"]
n_frag     = len(fragments)

fig = plt.figure(figsize=(22, 4.8 * n_frag), facecolor="white")
gs  = GridSpec(n_frag, 1, figure=fig,
               hspace=0.55, left=0.06, right=0.97,
               top=0.94, bottom=0.06)

fig.suptitle(
    "Adsorption Energy of Functional Group Proxies on N-doped TiO₂(111)\n"
    r"$E_{\mathrm{ad}} = E_{\mathrm{complex}} - E_{\mathrm{slab}} - E_{\mathrm{gas}}$"
    "   |   $\\Delta E_{\\mathrm{ad}} = E_{\\mathrm{ad}}(\\mathrm{N\\text{-}doped})"
    " - E_{\\mathrm{ad}}(\\mathrm{pristine})$"
    "   |   ★ = best N-doped config",
    fontsize=12, fontweight="bold", y=0.975,
)

for row, frag in enumerate(fragments):
    ax = fig.add_subplot(gs[row])

    subset = df_all[df_all["fragment"] == frag].copy()
    e_ref  = subset.loc[subset["slab_key"] == "pristine",
                        "E_ad (eV)"].values
    e_ref  = float(e_ref[0]) if len(e_ref) else 0.0
    e_gas  = subset["E_gas (eV)"].iloc[0] if "E_gas (eV)" in subset.columns else None

    keys   = [k for k in slab_order if k in subset["slab_key"].values]
    slabs  = subset.set_index("slab_key")

    e_vals  = [float(slabs.loc[k, "E_ad (eV)"])   for k in keys]
    d_vals  = [float(slabs.loc[k, "delta_E_ad"])   for k in keys]
    types   = [str(slabs.loc[k,  "slab_type"])     for k in keys]
    convs   = [bool(slabs.loc[k, "converged"])     for k in keys]
    colors  = [TYPE_CFG.get(t, {"color":"#6B7280"})["color"] for t in types]

    x      = np.arange(len(keys))
    BAR_W  = 0.62

    # ── Bars ──────────────────────────────────────────────────────
    bars = ax.bar(
        x, e_vals,
        color=colors, edgecolor="white",
        linewidth=0.4, width=BAR_W,
        zorder=3,
    )

    # Subtle dark edge for converged, red hatching for not converged
    for bar, conv, typ in zip(bars, convs, types):
        if not conv:
            bar.set_hatch("///")
            bar.set_edgecolor("#EF4444")
            bar.set_linewidth(0.8)

    # ── Pristine reference line ───────────────────────────────────
    ax.axhline(
        e_ref, color="#374151", ls=(0, (6, 3)),
        lw=1.3, alpha=0.85, zorder=4,
        label=f"Pristine ref  {e_ref:+.3f} eV",
    )

    # ── Zero line ─────────────────────────────────────────────────
    ax.axhline(0, color="#9CA3AF", lw=0.6, zorder=2)

    # ── Value labels ──────────────────────────────────────────────
    ymin, ymax = min(e_vals) * 1.18, max(e_vals) * 1.18 + 0.01
    ax.set_ylim(ymin, ymax)

    for bar, val, delta, conv in zip(bars, e_vals, d_vals, convs):
        bx   = bar.get_x() + bar.get_width() / 2
        sign = -1 if val < 0 else 1
        pad  = abs(ymax - ymin) * 0.012 * sign

        ax.text(
            bx, val + pad,
            f"{val:+.2f}",
            ha="center",
            va="top" if val < 0 else "bottom",
            fontsize=6.2, rotation=90,
            color="white" if abs(val) > abs(ymax - ymin) * 0.3 else "#111827",
            fontweight="500",
        )
        # ΔE_ad annotation (skip pristine)
        if delta != 0.0:
            clr = "#16A34A" if delta < 0 else "#DC2626"
            ax.text(
                bx, ymax * 0.97,
                f"Δ{delta:+.2f}",
                ha="center", va="top",
                fontsize=5.5,
                color=clr, style="italic",
            )

        # Not-converged marker
        if not conv:
            ax.text(
                bx, ymin * 0.95,
                "✗", ha="center", va="bottom",
                fontsize=8, color="#EF4444",
            )

    # ── Best N-doped highlight ────────────────────────────────────
    ndoped_idx = [(i, v) for i, (k, v) in
                  enumerate(zip(keys, e_vals)) if k != "pristine"]
    if ndoped_idx:
        best_i, best_v = min(ndoped_idx, key=lambda t: t[1])
        bars[best_i].set_edgecolor("#FBBF24")
        bars[best_i].set_linewidth(2.2)
        ax.text(
            x[best_i], best_v,
            "★", ha="center",
            va="bottom" if best_v >= 0 else "top",
            fontsize=12, color="#F59E0B", zorder=6,
        )

    # ── Type-group separators ─────────────────────────────────────
    prev_type = None
    for xi, t in enumerate(types):
        if t != prev_type and xi > 0:
            ax.axvline(xi - 0.5, color="#D1D5DB",
                       ls="-", lw=0.8, zorder=1)
        prev_type = t

    # ── Type labels (centered above each group) ───────────────────
    groups = {}
    for xi, t in enumerate(types):
        groups.setdefault(t, []).append(xi)
    for t, idxs in groups.items():
        mid = np.mean(idxs)
        ax.text(
            mid, ymax,
            TYPE_CFG.get(t, {}).get("label", t),
            ha="center", va="bottom",
            fontsize=7.5, color="#374151",
            fontweight="bold",
        )

    # ── Axes cosmetics ────────────────────────────────────────────
    ax.set_xticks(x)
    ax.set_xticklabels(
        [k.replace("_", "-") for k in keys],
        rotation=55, ha="right", fontsize=7,
    )
    ax.set_ylabel("$E_{\\mathrm{ad}}$ (eV)", fontsize=9)
    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter("%+.1f"))
    ax.grid(True, axis="y", color="#E5E7EB", linewidth=0.6, zorder=0)
    ax.set_axisbelow(True)

    # Subtitle with E_gas info
    egas_str = f"   |   $E_{{\\mathrm{{gas}}}}$ = {e_gas:+.3f} eV" if e_gas is not None else ""
    ax.set_title(
        f"({chr(96 + row + 1)})  {frag}  —  {ALL_PROXIES[frag]['label']}"
        f"  [{ALL_PROXIES[frag]['represents']}]{egas_str}",
        fontsize=9.5, fontweight="bold",
        loc="left", pad=4, color="#111827",
    )
    ax.legend(fontsize=7.5, loc="upper right",
              framealpha=0.85, edgecolor="#D1D5DB")

# ── Global legend ─────────────────────────────────────────────────
legend_patches = []
for t, cfg in TYPE_CFG.items():
    if t in df_all["slab_type"].values:
        legend_patches.append(
            mpatches.Patch(
                facecolor=cfg["color"],
                edgecolor="white",
                label=cfg["label"],
                linewidth=0.5,
            )
        )
legend_patches.append(
    mpatches.Patch(
        facecolor="white", edgecolor="#EF4444",
        hatch="///", label="Not converged (fmax > 0.05 eV/Å)",
        linewidth=0.8,
    )
)
fig.legend(
    handles=legend_patches,
    loc="lower center", ncol=len(legend_patches),
    fontsize=8.5, framealpha=0.9,
    edgecolor="#D1D5DB", bbox_to_anchor=(0.5, 0.002),
)

plt.savefig(
    "data/adsorption/all_adsorption_comparison.png",
    dpi=180, bbox_inches="tight", facecolor="white",
)
plt.show()
print("Saved → data/adsorption/all_adsorption_comparison.png")

## 10.8 Heatmap